# Decision Tree — Titanic Mini Capstone

Notebook này chỉ đọc shared artifacts từ common pipeline. Không scale dữ liệu, không thay đổi feature engineering, shared split, seed hoặc metric chính.


## 1. Thiết lập, đọc và kiểm tra data contract

In [1]:
from pathlib import Path
import json, time, warnings
import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.base import clone
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import StratifiedKFold, cross_validate, GridSearchCV, learning_curve
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report, roc_curve
)

RANDOM_STATE = 42
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "PROJECT_PLAN.md").exists())
COMMON_DIR = ROOT / "artifacts" / "common"
ARTIFACT_DIR = ROOT / "artifacts" / "decision_tree"
FIGURE_DIR = ROOT / "reports" / "figures" / "decision_tree"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

manifest = json.loads((COMMON_DIR / "feature_manifest.json").read_text(encoding="utf-8"))
X_train = pd.read_csv(COMMON_DIR / "X_train.csv")
X_valid = pd.read_csv(COMMON_DIR / "X_valid.csv")
X_full = pd.read_csv(COMMON_DIR / "X_full.csv")
X_test = pd.read_csv(COMMON_DIR / "X_test.csv")
y_train = pd.read_csv(COMMON_DIR / "y_train.csv").squeeze("columns")
y_valid = pd.read_csv(COMMON_DIR / "y_valid.csv").squeeze("columns")
y_full = pd.read_csv(COMMON_DIR / "y_full.csv").squeeze("columns")
test_passenger_id = pd.read_csv(COMMON_DIR / "test_passenger_id.csv")

feature_names = manifest["features"]
frames = {"X_train": X_train, "X_valid": X_valid, "X_full": X_full, "X_test": X_test}
expected_shapes = {"X_train": (712, 28), "X_valid": (179, 28), "X_full": (891, 28), "X_test": (418, 28)}
assert manifest["random_state"] == RANDOM_STATE
assert manifest["split"] == {"method":"train_test_split","test_size":0.2,"stratify":"Survived","random_state":42}
assert manifest["cv"]["method"] == "StratifiedKFold"
assert manifest["cv"]["n_splits"] == 10 and manifest["cv"]["shuffle"] is True
assert manifest["cv"]["random_state"] == RANDOM_STATE and manifest["cv"]["primary_metric"] == "accuracy"
for name, frame in frames.items():
    assert frame.shape == expected_shapes[name]
    assert frame.columns.tolist() == feature_names
    assert all(str(frame[c].dtype) == manifest["dtypes"][c] for c in feature_names)
    assert not frame.isna().any().any()
    assert np.isfinite(frame.to_numpy(dtype=float)).all()
assert y_train.name == y_valid.name == y_full.name == manifest["target"] == "Survived"
assert set(y_train.unique()) == set(y_valid.unique()) == set(y_full.unique()) == {0, 1}
assert len(y_train) == len(X_train) and len(y_valid) == len(X_valid) and len(y_full) == len(X_full)
assert test_passenger_id.columns.tolist() == ["PassengerId"]
assert len(test_passenger_id) == len(X_test) and test_passenger_id["PassengerId"].is_unique
CV = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)
print("Data contract hợp lệ.")
print({name: frame.shape for name, frame in frames.items()})


Data contract hợp lệ.
{'X_train': (712, 28), 'X_valid': (179, 28), 'X_full': (891, 28), 'X_test': (418, 28)}


## 2. Baseline

Baseline dùng đúng `DecisionTreeClassifier(random_state=42)`. Cross-validation chỉ chạy trên shared training split; validation không tham gia chọn mô hình.


In [2]:
def evaluate(model, X, y):
    pred = model.predict(X)
    score = model.predict_proba(X)[:, 1]
    return {
        "accuracy": float(accuracy_score(y, pred)),
        "precision": float(precision_score(y, pred, zero_division=0)),
        "recall": float(recall_score(y, pred, zero_division=0)),
        "f1": float(f1_score(y, pred, zero_division=0)),
        "roc_auc": float(roc_auc_score(y, score)),
        "confusion_matrix": confusion_matrix(y, pred).tolist(),
        "classification_report": classification_report(y, pred, output_dict=True, zero_division=0),
    }, pred, score

baseline = DecisionTreeClassifier(random_state=RANDOM_STATE)
baseline_cv = cross_validate(
    baseline, X_train, y_train, cv=CV, scoring="accuracy",
    return_train_score=True, n_jobs=1
)
baseline_cv_mean = float(baseline_cv["test_score"].mean())
baseline_cv_std = float(baseline_cv["test_score"].std())
baseline_cv_train_mean = float(baseline_cv["train_score"].mean())
baseline.fit(X_train, y_train)
baseline_train_accuracy = float(baseline.score(X_train, y_train))
baseline_metrics, baseline_pred, baseline_score = evaluate(baseline, X_valid, y_valid)

print(f"Baseline CV accuracy: {baseline_cv_mean:.4f} ± {baseline_cv_std:.4f}")
print(f"Baseline CV train accuracy: {baseline_cv_train_mean:.4f}")
print(classification_report(y_valid, baseline_pred, zero_division=0))
display(pd.DataFrame([{k:v for k,v in baseline_metrics.items() if k not in ["confusion_matrix","classification_report"]}], index=["baseline"]))


Baseline CV accuracy: 0.7893 ± 0.0324
Baseline CV train accuracy: 0.9855
              precision    recall  f1-score   support

           0       0.85      0.85      0.85       110
           1       0.76      0.77      0.76        69

    accuracy                           0.82       179
   macro avg       0.81      0.81      0.81       179
weighted avg       0.82      0.82      0.82       179



,accuracy,precision,recall,f1,roc_auc
baseline,0.815642,0.757143,0.768116,0.76259,0.801976


## 3. GridSearchCV

Khảo sát `criterion`, `max_depth`, `min_samples_split`, `min_samples_leaf`, `max_features` và `class_weight`. `log_loss` chỉ được thêm sau khi kiểm tra phiên bản hiện tại có hỗ trợ.


In [3]:
criteria = ["gini", "entropy"]
try:
    DecisionTreeClassifier(criterion="log_loss", random_state=RANDOM_STATE).fit(X_train.iloc[:20], y_train.iloc[:20])
    criteria.append("log_loss")
except (ValueError, TypeError):
    print("criterion='log_loss' không được hỗ trợ; bỏ khỏi grid.")

param_grid = {
    "criterion": criteria,
    "max_depth": [None, 2, 3, 4, 5, 6, 8, 10],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 4, 8, 12],
    "max_features": [None, "sqrt", "log2"],
    "class_weight": [None, "balanced"],
}
n_candidates = int(np.prod([len(v) for v in param_grid.values()]))
print(f"Grid: {n_candidates:,} cấu hình × 10 folds = {n_candidates*10:,} fits")
grid_search = GridSearchCV(
    DecisionTreeClassifier(random_state=RANDOM_STATE),
    param_grid=param_grid, scoring="accuracy", cv=CV, n_jobs=1,
    return_train_score=True, refit=True, verbose=1
)
t0 = time.perf_counter()
grid_search.fit(X_train, y_train)
tuning_seconds = float(time.perf_counter() - t0)

cv_results = pd.DataFrame(grid_search.cv_results_).sort_values(
    ["rank_test_score", "std_test_score"], kind="stable"
)
cv_results.to_csv(ARTIFACT_DIR / "cv_results.csv", index=False)
best_index = int(grid_search.best_index_)
best_cv_std = float(grid_search.cv_results_["std_test_score"][best_index])
best_cv_train_mean = float(grid_search.cv_results_["mean_train_score"][best_index])
print(f"Tuning time: {tuning_seconds:.2f} giây")
print(f"Best params: {grid_search.best_params_}")
print(f"Best CV accuracy: {grid_search.best_score_:.4f} ± {best_cv_std:.4f}")
top_cols = [
    "rank_test_score","mean_test_score","std_test_score","mean_train_score","mean_fit_time",
    "param_criterion","param_max_depth","param_min_samples_split","param_min_samples_leaf",
    "param_max_features","param_class_weight"
]
display(cv_results[top_cols].head(10).reset_index(drop=True))


Grid: 2,880 cấu hình × 10 folds = 28,800 fits
Fitting 10 folds for each of 2880 candidates, totalling 28800 fits


Tuning time: 250.22 giây
Best params: {'class_weight': None, 'criterion': 'gini', 'max_depth': None, 'max_features': None, 'min_samples_leaf': 8, 'min_samples_split': 2}
Best CV accuracy: 0.8384 ± 0.0396


,rank_test_score,mean_test_score,std_test_score,mean_train_score,mean_fit_time,param_criterion,param_max_depth,param_min_samples_split,param_min_samples_leaf,param_max_features,param_class_weight
0,1,0.838439,0.038052,0.873129,0.003812,gini,8,2,8,None,None
1,1,0.838439,0.038052,0.873129,0.003801,gini,8,5,8,None,None
2,1,0.838439,0.038052,0.873129,0.003717,gini,8,10,8,None,None
3,1,0.838439,0.039586,0.873909,0.003693,gini,None,2,8,None,None
4,1,0.838439,0.039586,0.873909,0.003720,gini,None,5,8,None,None
5,1,0.838439,0.039586,0.873909,0.003712,gini,None,10,8,None,None
6,1,0.838439,0.039586,0.873909,0.003728,gini,10,2,8,None,None
7,1,0.838439,0.039586,0.873909,0.003814,gini,10,5,8,None,None
8,1,0.838439,0.039586,0.873909,0.003832,gini,10,10,8,None,None
9,10,0.835681,0.033888,0.874065,0.003693,gini,6,10,4,None,None


## 4. Hold-out, overfitting và learning curve

In [4]:
best_train_model = grid_search.best_estimator_
tuned_train_accuracy = float(best_train_model.score(X_train, y_train))
tuned_metrics, tuned_pred, tuned_score = evaluate(best_train_model, X_valid, y_valid)
baseline_gap = baseline_cv_train_mean - baseline_cv_mean
tuned_gap = best_cv_train_mean - float(grid_search.best_score_)

comparison = pd.DataFrame({
    "baseline": {
        "train_accuracy": baseline_train_accuracy, "cv_train_mean": baseline_cv_train_mean,
        "cv_mean": baseline_cv_mean, "cv_std": baseline_cv_std,
        **{"valid_"+k:v for k,v in baseline_metrics.items() if k not in ["confusion_matrix","classification_report"]},
    },
    "tuned": {
        "train_accuracy": tuned_train_accuracy, "cv_train_mean": best_cv_train_mean,
        "cv_mean": float(grid_search.best_score_), "cv_std": best_cv_std,
        **{"valid_"+k:v for k,v in tuned_metrics.items() if k not in ["confusion_matrix","classification_report"]},
    },
})
display(comparison)
print(classification_report(y_valid, tuned_pred, zero_division=0))
print(f"Train–CV gap: baseline={baseline_gap:.4f}, tuned={tuned_gap:.4f}")
if tuned_gap < baseline_gap:
    print("Regularization đã giảm chênh lệch train–CV, cho thấy overfitting được kiểm soát tốt hơn.")
else:
    print("Tuned model chưa giảm chênh lệch train–CV; cần thận trọng với overfitting.")

fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))
for ax, matrix, title in [
    (axes[0], baseline_metrics["confusion_matrix"], "Baseline confusion matrix"),
    (axes[1], tuned_metrics["confusion_matrix"], "Tuned confusion matrix"),
]:
    sns.heatmap(matrix, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax)
    ax.set(xlabel="Predicted label", ylabel="True label", title=title)
for score, label in [(baseline_score, "Baseline"), (tuned_score, "Tuned")]:
    fpr, tpr, _ = roc_curve(y_valid, score)
    axes[2].plot(fpr, tpr, lw=2, label=f"{label} (AUC={roc_auc_score(y_valid, score):.3f})")
axes[2].plot([0,1],[0,1],"k--",alpha=.6)
axes[2].set(xlabel="False positive rate",ylabel="True positive rate",title="Hold-out ROC curve")
axes[2].legend()
fig.tight_layout()
fig.savefig(FIGURE_DIR / "validation_diagnostics.png", dpi=160, bbox_inches="tight")
plt.show()

train_sizes, lc_train, lc_valid = learning_curve(
    clone(best_train_model), X_train, y_train, train_sizes=np.linspace(.1,1,10),
    cv=CV, scoring="accuracy", n_jobs=1, shuffle=True, random_state=RANDOM_STATE
)
tr_mean, tr_std = lc_train.mean(axis=1), lc_train.std(axis=1)
va_mean, va_std = lc_valid.mean(axis=1), lc_valid.std(axis=1)
fig, ax = plt.subplots(figsize=(8,5.5))
ax.plot(train_sizes,tr_mean,"o-",label="Training accuracy")
ax.fill_between(train_sizes,tr_mean-tr_std,tr_mean+tr_std,alpha=.18)
ax.plot(train_sizes,va_mean,"o-",label="CV accuracy")
ax.fill_between(train_sizes,va_mean-va_std,va_mean+va_std,alpha=.18)
ax.set(xlabel="Training examples",ylabel="Accuracy",title="Tuned Decision Tree learning curve")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURE_DIR / "learning_curve.png", dpi=160, bbox_inches="tight")
plt.show()


,baseline,tuned
train_accuracy,0.984551,0.875000
cv_train_mean,0.985486,0.873909
cv_mean,0.789280,0.838439
cv_std,0.032359,0.039586
valid_accuracy,0.815642,0.832402
valid_precision,0.757143,0.800000
valid_recall,0.768116,0.753623
valid_f1,0.762590,0.776119
valid_roc_auc,0.801976,0.859552


              precision    recall  f1-score   support

           0       0.85      0.88      0.87       110
           1       0.80      0.75      0.78        69

    accuracy                           0.83       179
   macro avg       0.83      0.82      0.82       179
weighted avg       0.83      0.83      0.83       179

Train–CV gap: baseline=0.1962, tuned=0.0355
Regularization đã giảm chênh lệch train–CV, cho thấy overfitting được kiểm soát tốt hơn.


C:\Users\Do Truong Son\AppData\Local\Temp\ipykernel_8208\2023526789.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\Do Truong Son\AppData\Local\Temp\ipykernel_8208\2023526789.py:59: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Cấu trúc cây và feature importance

In [5]:
tree_depth = int(best_train_model.get_depth())
tree_leaves = int(best_train_model.get_n_leaves())
importance = pd.DataFrame({
    "feature": feature_names,
    "importance": best_train_model.feature_importances_,
}).sort_values("importance", ascending=False, ignore_index=True)
print(f"Tuned tree depth: {tree_depth}; leaves: {tree_leaves}")
display(importance)

shown = importance.query("importance > 0").sort_values("importance")
fig, ax = plt.subplots(figsize=(10,7))
ax.barh(shown["feature"],shown["importance"],color="#4C78A8")
ax.set(xlabel="Impurity-based importance",title="Decision Tree feature importance")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "feature_importance.png",dpi=160,bbox_inches="tight")
plt.show()

shown_depth = min(3, tree_depth)
fig, ax = plt.subplots(figsize=(24,10))
plot_tree(
    best_train_model, feature_names=feature_names,
    class_names=["Not survived","Survived"], filled=True, rounded=True,
    max_depth=shown_depth, fontsize=8, ax=ax
)
ax.set_title(f"Tuned Decision Tree — first {shown_depth} levels")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "best_tree_first_3_levels.png",dpi=160,bbox_inches="tight")
plt.show()


Tuned tree depth: 8; leaves: 49


,feature,importance
0,Title_Mr,0.487264
1,Fare,0.142245
2,Pclass,0.120294
3,FamilyCategory_Large,0.063799
4,Age,0.049366
5,Title_Rare,0.047569
6,AgeBand_1,0.042498
7,FareBand_1,0.016178
8,Embarked_S,0.009416
9,FareBand_2,0.009329


C:\Users\Do Truong Son\AppData\Local\Temp\ipykernel_8208\3541797231.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\Do Truong Son\AppData\Local\Temp\ipykernel_8208\3541797231.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Refit full labeled data và xuất artifact

Cấu hình tốt nhất được chọn chỉ từ training CV. Sau đánh giá hold-out, cùng cấu hình được refit trên toàn bộ 891 bản ghi có nhãn.


In [6]:
final_model = clone(best_train_model).fit(X_full, y_full)
test_pred = final_model.predict(X_test).astype(int)
test_score = final_model.predict_proba(X_test)[:,1]

validation_predictions = pd.DataFrame({
    "row_id": X_valid.index,
    "y_true": y_valid.to_numpy(dtype=int),
    "y_pred": tuned_pred.astype(int),
    "y_score": tuned_score,
})
test_predictions = pd.DataFrame({
    "PassengerId": test_passenger_id["PassengerId"].to_numpy(dtype=int),
    "Survived": test_pred,
    "score": test_score,
})
submission = test_predictions[["PassengerId","Survived"]].copy()

joblib.dump(final_model, ARTIFACT_DIR / "best_model.joblib")
validation_predictions.to_csv(ARTIFACT_DIR / "validation_predictions.csv",index=False)
test_predictions.to_csv(ARTIFACT_DIR / "test_predictions.csv",index=False)
submission.to_csv(ARTIFACT_DIR / "submission.csv",index=False)

metrics = {
    "model_name":"DecisionTreeClassifier","random_state":RANDOM_STATE,"cv_folds":10,
    "primary_metric":"accuracy","grid_candidates":n_candidates,"grid_total_fits":n_candidates*10,
    "tuning_seconds":tuning_seconds,
    "baseline_cv_mean":baseline_cv_mean,"baseline_cv_std":baseline_cv_std,
    "baseline_cv_train_mean":baseline_cv_train_mean,"baseline_train_accuracy":baseline_train_accuracy,
    "baseline_validation_accuracy":baseline_metrics["accuracy"],
    "baseline_validation_precision":baseline_metrics["precision"],
    "baseline_validation_recall":baseline_metrics["recall"],
    "baseline_validation_f1":baseline_metrics["f1"],
    "baseline_validation_roc_auc":baseline_metrics["roc_auc"],
    "baseline_confusion_matrix":baseline_metrics["confusion_matrix"],
    "baseline_classification_report":baseline_metrics["classification_report"],
    "best_cv_score":float(grid_search.best_score_),"best_cv_std":best_cv_std,
    "best_cv_train_mean":best_cv_train_mean,"best_params":grid_search.best_params_,
    "tuned_train_accuracy":tuned_train_accuracy,
    "validation_accuracy":tuned_metrics["accuracy"],
    "validation_precision":tuned_metrics["precision"],
    "validation_recall":tuned_metrics["recall"],
    "validation_f1":tuned_metrics["f1"],
    "validation_roc_auc":tuned_metrics["roc_auc"],
    "validation_confusion_matrix":tuned_metrics["confusion_matrix"],
    "validation_classification_report":tuned_metrics["classification_report"],
    "tree_depth_train_split":tree_depth,"tree_leaves_train_split":tree_leaves,
    "final_tree_depth_full_data":int(final_model.get_depth()),
    "final_tree_leaves_full_data":int(final_model.get_n_leaves()),
    "feature_importance":{r.feature:float(r.importance) for r in importance.itertuples()},
    "overfitting":{
        "baseline_train_cv_gap":float(baseline_gap),
        "tuned_train_cv_gap":float(tuned_gap),
        "gap_reduction":float(baseline_gap-tuned_gap),
    },
}
(ARTIFACT_DIR / "metrics.json").write_text(json.dumps(metrics,indent=2,ensure_ascii=False),encoding="utf-8")

expected = ["best_model.joblib","metrics.json","cv_results.csv","validation_predictions.csv","test_predictions.csv","submission.csv"]
assert all((ARTIFACT_DIR / f).exists() for f in expected)
assert validation_predictions.columns.tolist() == ["row_id","y_true","y_pred","y_score"]
assert test_predictions.columns.tolist() == ["PassengerId","Survived","score"]
assert submission.columns.tolist() == ["PassengerId","Survived"] and len(submission) == len(X_test)
assert set(submission["Survived"].unique()).issubset({0,1})
assert validation_predictions["y_score"].between(0,1).all() and test_predictions["score"].between(0,1).all()

print("Đã xuất đủ sáu artifact:", expected)
print(f"Best CV: {grid_search.best_score_:.4f} ± {best_cv_std:.4f}")
print(f"Hold-out: accuracy={tuned_metrics['accuracy']:.4f}, precision={tuned_metrics['precision']:.4f}, recall={tuned_metrics['recall']:.4f}, F1={tuned_metrics['f1']:.4f}, ROC-AUC={tuned_metrics['roc_auc']:.4f}")
print(f"Tuned tree: depth={tree_depth}, leaves={tree_leaves}; train–CV gap giảm {baseline_gap-tuned_gap:.4f}.")
print("Hạn chế: impurity-based importance có thể thiên lệch; một shared hold-out split vẫn có độ bất định.")


Đã xuất đủ sáu artifact: ['best_model.joblib', 'metrics.json', 'cv_results.csv', 'validation_predictions.csv', 'test_predictions.csv', 'submission.csv']
Best CV: 0.8384 ± 0.0396
Hold-out: accuracy=0.8324, precision=0.8000, recall=0.7536, F1=0.7761, ROC-AUC=0.8596
Tuned tree: depth=8, leaves=49; train–CV gap giảm 0.1607.
Hạn chế: impurity-based importance có thể thiên lệch; một shared hold-out split vẫn có độ bất định.
